In [0]:
import json
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

# 1. WATERMARK: Only pull rows from Bronze we haven't processed yet
try:
    last_processed_ts = spark.sql("SELECT max(processing_timestamp) FROM nasa_project.silver.asteroids_cleaned").collect()[0][0]
    if last_processed_ts is None: last_processed_ts = datetime(1900, 1, 1)
except:
    last_processed_ts = datetime(1900, 1, 1)

new_raw_rows = spark.table("nasa_project.bronze.asteroids_raw").filter(F.col("ingested_at") > last_processed_ts).collect()

if not new_raw_rows:
    dbutils.notebook.exit("Success: No new raw data to process.")

# 2. FLATTEN: Process only the new batches
flattened_rows = []
for row in new_raw_rows:
    data = json.loads(row["raw_json"].replace("'", '"').replace("False", "false").replace("True", "true"))
    neo_data = data.get("near_earth_objects", {})
    
    for date, asteroids in neo_data.items():
        for a in asteroids:
            flattened_rows.append({
                "date": date,
                "name": a.get("name"),
                "is_hazardous": a.get("is_potentially_hazardous_asteroid"),
                "size_km": a.get("estimated_diameter", {}).get("kilometers", {}).get("estimated_diameter_max"),
                "miss_km": float(a.get("close_approach_data", [{}])[0].get("miss_distance", {}).get("kilometers", 0)),
                "processing_timestamp": row["ingested_at"] # Key for incremental logic
            })

silver_df = spark.createDataFrame(flattened_rows).withColumn("date", F.col("date").cast("date"))

# 3. MERGE: Upsert into Silver (Hired-ready skill!)
target_silver = "nasa_project.silver.asteroids_cleaned"
if spark.catalog.tableExists(target_silver):
    dt = DeltaTable.forName(spark, target_silver)
    dt.alias("t").merge(silver_df.alias("s"), "t.date = s.date AND t.name = s.name") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    silver_df.write.mode("overwrite").saveAsTable(target_silver)